In [1]:
from pathlib import Path
from ultralytics import YOLO
import cv2
import shutil

In [2]:
BASE_DIR = Path(r'D:\Datasets\Chocolate Classification')
CLASSES = ['dark', 'white']
IMG_SUFFIXS = ['.jpg', '.png']

In [7]:
train_txt = Path.read_text(BASE_DIR / 'train.txt')
train_txt = train_txt.replace('data/obj', 'obj')
train_txt_list = train_txt.split("\n")

In [7]:
move_data = BASE_DIR / 'obj_train_data'

In [8]:
move_data_txt = [move_data_dir for move_data_dir in move_data.iterdir() if move_data_dir.suffix == '.txt']
move_data_img = [move_data_dir for move_data_dir in move_data.iterdir() if move_data_dir.suffix in IMG_SUFFIXS]

In [33]:
data_dir = BASE_DIR / 'train'
images_dir = data_dir / 'images'

In [70]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    move_data_img, 
    test_size=0.2,
    random_state=42
)

In [78]:
train_data[0].name.replace('.jpg', '.txt')

'Image_211.txt'

In [80]:
def prepare_data(data, mode):
    data_dir = BASE_DIR / mode
    move_data = BASE_DIR / 'obj_train_data'
    images_dir = data_dir / 'images'
    labels_dir = data_dir / 'labels'
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)

    for img_dir in data:
        
        shutil.copy(move_data / img_dir.name, 
                    images_dir / img_dir.name)
        
        txt_dir = img_dir.name.replace('.jpg', '.txt')
        
        shutil.copy(move_data / txt_dir, 
                    labels_dir / txt_dir)



prepare_data(train_data, 'train')
prepare_data(val_data, 'val')


In [82]:
with (BASE_DIR / 'data.yaml').open("w", encoding ="utf-8") as f:
            f.write(f'''
path: {BASE_DIR}
# Имена подпапок
train: train/images
val: val/images

# Число классов
nc: {len(CLASSES)}

names:
  0: {CLASSES[0]}
  1: {CLASSES[1]}       

''')

In [84]:
model_dir = Path.cwd().parent / 'models'
model_dir

WindowsPath('c:/Users/ASUS/Documents/GitHub/CV-training/.venv/models')

In [208]:
model = YOLO(model_dir / "yolo11n.pt")

In [3]:
model_dir = Path.cwd()
model = YOLO(model_dir / "best.pt")

In [209]:
results = model.train(
    data=BASE_DIR / 'data.yaml',  # путь к вашему data.yaml
    epochs=50,                      # число эпох
    imgsz=480,                      # размер изображения
    batch=16,                       # размер батча (уменьшите, если не хватает памяти)
    device='cpu',                  # 'cpu' или 'cuda' или 0,1 [список GPU]
    project=model_dir,
    patience=100,
    workers=4,                      # число workers для загрузки данных
    lr0=0.01,                       # начальный learning rate
    pretrained=True,                # использовать предобученные веса
    optimizer='auto',                 # или 'AdamW'
    hsv_v = 0.8,
    degrees = 90
) 

New https://pypi.org/project/ultralytics/8.3.227 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.225  Python-3.13.9 torch-2.9.0+cpu CPU (Intel Core i7-8700T 2.40GHz)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Datasets\Chocolate Classification\data.yaml, degrees=90, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.8, imgsz=480, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=c:\Users\ASUS\Documents\GitHub\CV-training\.venv\chocolate\yolo training\yolo11n.pt

In [4]:
model_dir = Path.cwd()
model = YOLO(model_dir / "best.pt")


In [18]:
def detect(frame):
    results = model(frame)
    frame_after = frame.copy()
    for i,result in enumerate (results):
        #print(result.boxes.xyxy, result.boxes.cls is list)
        xyxy = result.boxes.xyxy
        for [x1,y1,x2,y2], cl in zip(xyxy, result.boxes.cls) :
            cv2.rectangle(frame_after, (int(x1),int(y1)), (int(x2),int(y2)), (255, 0, 0), 3)
            cv2.putText(frame_after, str(CLASSES[int(cl)]), (int(x1)+10, int(y1)+30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.8, (30, 30, 30), 3)
    return frame_after

In [19]:
import random

#img_path = str(Path(r"D:\Datasets\Chocolate Classification\obj_train_data\Image_21.jpg"))
#img_path = str(Path(r"D:\Datasets\Chocolate Classification\obj_train_data\Image_1.jpg"))
img_path = str(random.choice(move_data_img))
print(img_path)
img = cv2.imread(img_path)
img = detect(img)
cv2.imshow('frame', img)
cv2.waitKey(0)
cv2.destroyAllWindows()

D:\Datasets\Chocolate Classification\obj_train_data\Image_163.jpg



0: 480x320 1 dark, 56.7ms
Speed: 2.0ms preprocess, 56.7ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 320)
